In [2]:
# =====================================================================
# 1. FIXED MOCK LLM CLIENT
# =====================================================================
class MockLLMClient:
    """Simulates LLM responses with robust string matching."""
    def generate(self, system_prompt: str, user_prompt: str) -> str:
        # Use lower-case matching to avoid keyword misses
        system_lower = system_prompt.lower()
        
        if "orchestrator" in system_lower:
            return json.dumps({
                "required_skills": ["Kubernetes", "FinTech Infrastructure", "Go/Java"],
                "target_companies": ["Stripe", "Adyen", "Revolut"],
                "sourcing_strategy": "Search for open-source contributors to CNCF projects working in financial sectors."
            }, indent=2)
            
        elif "sourcing" in system_lower:  # <-- Fixed string matching
            return json.dumps([
                {"name": "Alex Rivera", "github_user": "alexr_dev", "current_role": "Senior DevOps at a Tier-1 Bank", "bio": "Maintaining high-throughput ledger infra. K8s contributor."},
                {"name": "Sam Taylor", "github_user": "samt_codes", "current_role": "Infrastructure Eng at PaymentsCorp", "bio": "Building reliable trading platforms. Docker and multi-cluster mesh enthusiast."}
            ], indent=2)
            
        elif "screener" in system_lower:
            name = "Alex" if "Alex" in user_prompt else "Sam"
            return json.dumps({
                "match_score": "95%" if name == "Alex" else "88%",
                "technical_verdict": f"Strong validation. Core focus matches the requested stack perfectly.",
                "key_highlight": "Active contributor to Kubernetes upstream network routing primitives." if name == "Alex" else "Deep knowledge of high-concurrency payment streams."
            }, indent=2)
            
        elif "copywriter" in system_lower:
            return json.dumps({  # <-- Wrapped in JSON to make parsing robust across the board
                "email": "Subject: Impressed by your upstream K8s contributions\n\nHi Candidate, I noticed your recent work..."
            }, indent=2)
            
        return '{"error": "Unknown system prompt instruction"}'


# =====================================================================
# 2. UPDATED WORKER AND COPYWRITER PARSING
# =====================================================================
class SourcingWorker:
    def __init__(self, client):
        self.client = client
        self.role = "Worker 1: Sourcing Specialist"
        # Explicitly keeping 'Sourcing' in the system prompt
        self.system_prompt = "You are an expert technical sourcing specialist. Extract raw candidate profiles."

    def execute(self, search_criteria: Dict[str, Any]) -> List[Dict[str, Any]]:
        print(f" -> [{self.role}] Scanning talent pools using criteria: {search_criteria.get('required_skills')}...")
        time.sleep(1)
        raw_response = self.client.generate(self.system_prompt, f"Criteria: {search_criteria}")
        return json.loads(raw_response)


class OutreachWorker:
    def __init__(self, client):
        self.client = client
        self.role = "Worker 3: Outreach Copywriter"
        self.system_prompt = "You are an elite talent acquisition copywriter."

    def execute(self, candidate: Dict[str, Any], technical_report: Dict[str, Any]) -> str:
        print(f" -> [{self.role}] Crafting hyper-customized messaging for {candidate['name']}...")
        time.sleep(0.5)
        user_input = f"Candidate: {candidate}\nReport: {technical_report}"
        raw_response = self.client.generate(self.system_prompt, user_input)
        # Safely parse the text or return raw if it isn't JSON structured
        try:
            return json.loads(raw_response).get("email", raw_response)
        except json.JSONDecodeError:
            return raw_response

In [3]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

In [4]:
load_dotenv()

# Initialize the client exactly like the lab setup
# (Will pull OPENAI_API_KEY natively from your .env file)
client = OpenAI()

In [6]:
# 1. DEFINE THE INPUT DATA
job_description = """
Role: Technical Account Manager
Requirements:
- 2+ years managing high-throughput infrastructure.
- Deep expertise with cloud infrastructure and security.
- Experience coaching junior engineers and working across teams.
"""

candidate_resume = """
Name: Jordan Vance
Experience:
- 2 years working for a cloud consultancy.
- Automated CI/CD pipelines, reducing deploy times by 40%.
- Conducted weekly architecture reviews and mentored 4 junior engineers.
- Highly analytical, and people friendly.
"""

In [7]:
# 2. DEFINE THE PARALLEL EXPERT AGENTS
# We create a dictionary of our different pioneer/expert personas
agents = {
    "Technical Specialist": (
        "You are a strict, no-nonsense Principal Infrastructure Architect. "
        "Evaluate the candidate strictly on hard skills, technical depth, scaling capabilities, and infrastructure metrics. "
        "Ignore fluff and evaluate if they can handle deep tech problems."
    ),
    "Talent & Culture Partner": (
        "You are an expert HR Talent Partner focused on culture, leadership, and team growth. "
        "Evaluate the candidate's experience with mentorship, cross-team collaboration, soft skills, and career progression."
    ),
    "Risk & Compliance Auditor": (
        "You are an HR Compliance officer. Evaluate if the candidate has structural background in security, "
        "operational boundaries, compliance, and if their tenure and history present any organizational risk patterns."
    )
}

In [ ]:
# 3. COLLECT RESPONSES IN PARALLEL
# This mirrors the lab logic: Loop through the agents, call the API, and store their outputs
agent_outputs = {}

print("--- Launching Parallel Expert Evaluations ---")
for agent_name, system_instruction in agents.items():
    print(f"Running evaluation via: {agent_name}...")
    
    prompt = f"Job Description:\n{job_description}\n\nCandidate Resume:\n{candidate_resume}"
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Using standard course-friendly models
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5
    )

    # Store the response text


--- Launching Parallel Expert Evaluations ---
Running evaluation via: Technical Specialist...
Running evaluation via: Talent & Culture Partner...
Running evaluation via: Risk & Compliance Auditor...


In [10]:
agent_outputs[agent_name] = response.choices[0].message.content

# 4. THE JUDGE AGENT EVALUATES AND RANKS THE OUTPUTS
print("\n--- Sending Evaluations to the Senior HR Director Judge ---")


--- Sending Evaluations to the Senior HR Director Judge ---


In [11]:
# Combine the outputs into a single text block for the judge
compiled_evaluations = ""
for agent_name, output in agent_outputs.items():
    compiled_evaluations += f"=== EVALUATION FROM {agent_name.upper()} ===\n{output}\n\n"

judge_system_prompt = """
You are a Senior HR Director. Your job is to review the independent assessments provided by three specialized recruiting agents.
Compare their perspectives, synthesize their points, and provide a final unified hiring recommendation.

You must output your final decision in strict JSON format with these exact keys:
{
  "synthesis": "Your analysis comparing the different perspectives of the experts",
  "final_verdict": "Proceed to Interview / Reject / Hold",
  "overall_score": "Score out of 100",
  "ranking_of_agent_perspectives": [
     "List the 3 agents in order of whose argument was most critical/persuasive for this specific candidate decision"
  ]
}
"""

In [12]:
judge_response = client.chat.completions.create(
    model="gpt-4o", # Using a stronger model for the reasoning/judgment layer
    messages=[
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": f"Here are the expert reviews:\n\n{compiled_evaluations}"}
    ],
    response_format={"type": "json_object"}, # Ensuring strict parsing safety
    temperature=0.2
)

# 5. PRINT THE FINAL RESULTS
final_judgment = json.loads(judge_response.choices[0].message.content)

print("\n" + "="*50)
print("FINAL SYNTHESIZED JUDGMENT")
print("="*50)
print(json.dumps(final_judgment, indent=2))


FINAL SYNTHESIZED JUDGMENT
{
  "synthesis": "The Risk & Compliance Auditor's evaluation highlights both strengths and potential gaps in Jordan Vance's qualifications. Jordan's experience in automating CI/CD pipelines and mentoring junior engineers suggests a solid understanding of operational boundaries and leadership potential. However, the lack of explicit security certifications and compliance experience is a concern, as these are critical for the Technical Account Manager role. The auditor suggests that an interview could further explore Jordan's depth of knowledge in these areas, which could help determine if the candidate can mitigate potential risks associated with security and compliance in cloud environments.",
  "final_verdict": "Proceed to Interview",
  "overall_score": "70",
  "ranking_of_agent_perspectives": [
    "Risk & Compliance Auditor"
  ]
}
